<a href="https://colab.research.google.com/github/JoelForson/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Lab%2013/Hedonic_Pricing_and_the_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/RyanPiao/ECON5200-Applied-Data-Analytics-in-Economics/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        18:46:23   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [6]:
df.head()

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54


In [15]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:49:42   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [19]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -1764.5082212704574


In [18]:
"""
=============================================================================
 AI Assisted 3D OLS REGRESSION HYPERPLANE VISUALIZER
  Multivariate: Sale_Price ~ Property_Age + Distance_to_Tech_Hub
  Engine: statsmodels OLS  |  Renderer: plotly.graph_objects
=============================================================================
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 ── Synthetic Dataset
#   Replace this block with your actual DataFrame if you have real data.
#   The variable names match the lab spec exactly.
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(7)
n = 300

Property_Age         = np.random.uniform(0, 60, n)      # years since built
Distance_to_Tech_Hub = np.random.uniform(0.5, 25, n)    # miles

# True data-generating process (DGP):
#   Each extra year of age  → -$1,800 in price (depreciation)
#   Each extra mile away    → -$4,500 in price (location premium decay)
Sale_Price = (
    850_000
    - 1_800  * Property_Age
    - 4_500  * Distance_to_Tech_Hub
    + np.random.normal(0, 35_000, n)    # stochastic noise
)

df = pd.DataFrame({
    "Sale_Price":            Sale_Price,
    "Property_Age":          Property_Age,
    "Distance_to_Tech_Hub":  Distance_to_Tech_Hub,
})

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 ── OLS Estimation via statsmodels
# ─────────────────────────────────────────────────────────────────────────────

X = df[["Property_Age", "Distance_to_Tech_Hub"]]
X = sm.add_constant(X)      # adds the 'const' column → β₀ (intercept)
y = df["Sale_Price"]

results = sm.OLS(y, X).fit()
print(results.summary())

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 ── Extract Coefficients from the Results Object
#
#   results.params is a pandas Series keyed by column name:
#     results.params["const"]                → β₀ (intercept)
#     results.params["Property_Age"]         → β₁ (slope on age)
#     results.params["Distance_to_Tech_Hub"] → β₂ (slope on distance)
#
#   The fitted hyperplane equation is therefore:
#     Ŷ = β₀ + β₁·(Property_Age) + β₂·(Distance_to_Tech_Hub)
#
#   We pull each coefficient by name so the code stays readable and immune
#   to column-order changes in the design matrix.
# ─────────────────────────────────────────────────────────────────────────────

beta_0 = results.params["const"]                 # intercept
beta_1 = results.params["Property_Age"]          # ∂Ŷ/∂(Property_Age)
beta_2 = results.params["Distance_to_Tech_Hub"]  # ∂Ŷ/∂(Distance_to_Tech_Hub)

print(f"\n📐 Estimated Hyperplane:")
print(f"   Sale_Price = {beta_0:,.0f}")
print(f"             + {beta_1:,.1f} × Property_Age")
print(f"             + {beta_2:,.1f} × Distance_to_Tech_Hub")
print(f"\n   R² = {results.rsquared:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 ── Build the Meshgrid for the Regression Surface
#
#   A 3D surface requires Z values at every (X, Y) grid point.
#   np.meshgrid() creates two 2D arrays from two 1D coordinate vectors:
#     - age_range  : 50 evenly-spaced values spanning Property_Age
#     - dist_range : 50 evenly-spaced values spanning Distance_to_Tech_Hub
#
#   meshgrid returns:
#     AGE_GRID  [50×50] — every row is identical (age varies across columns)
#     DIST_GRID [50×50] — every column is identical (dist varies down rows)
#
#   Together they define 50×50 = 2,500 (age, dist) coordinate pairs.
#   Applying the OLS equation elementwise across both grids gives the
#   predicted Sale_Price surface — the visual regression hyperplane.
# ─────────────────────────────────────────────────────────────────────────────

# 1D range vectors — span the observed data with a little buffer
age_range  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(), 50)

dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(), 50)

# 2D coordinate grids from the 1D vectors
AGE_GRID, DIST_GRID = np.meshgrid(age_range, dist_range)

# Predicted surface: apply the OLS equation across every grid cell
#   Shape: (50, 50) — one predicted value per (age, dist) pair
Z_SURFACE = beta_0 + beta_1 * AGE_GRID + beta_2 * DIST_GRID

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 ── Residual Colouring for the Scatter Points
#   Points above the plane → warm (over-prediction)
#   Points below the plane → cool (under-prediction)
# ─────────────────────────────────────────────────────────────────────────────

fitted  = results.fittedvalues          # ŷᵢ from statsmodels
resid   = results.resid                 # eᵢ = yᵢ − ŷᵢ

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 ── Compose the 3D Figure with plotly.graph_objects
# ─────────────────────────────────────────────────────────────────────────────

fig = go.Figure()

# ── Layer 1: Regression Hyperplane (Surface) ─────────────────────────────────
fig.add_trace(go.Surface(
    x=AGE_GRID,
    y=DIST_GRID,
    z=Z_SURFACE,
    colorscale=[
        [0.0, "#0d2137"],   # deep navy  → low predicted price
        [0.5, "#1a5e8a"],   # mid teal
        [1.0, "#00e5c8"],   # electric cyan → high predicted price
    ],
    opacity=0.55,
    showscale=True,
    colorbar=dict(
        title=dict(text="Predicted<br>Sale Price ($)", font=dict(color="#aaccdd", size=11)),
        tickfont=dict(color="#aaccdd", size=10),
        x=1.02,
    ),
    name="OLS Hyperplane",
    hovertemplate=(
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} mi<br>"
        "Ŷ: $%{z:,.0f}<extra>Regression Plane</extra>"
    ),
    contours=dict(
        z=dict(show=True, usecolormap=True, highlightcolor="white", project_z=True)
    ),
))

# ── Layer 2: Actual Data Points coloured by residual sign ────────────────────
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=4.5,
        color=resid,                        # residual magnitude drives colour
        colorscale=[
            [0.0,  "#DC143C"],              # crimson   → large negative residual
            [0.5,  "#f0f0f0"],              # white     → on the plane (e ≈ 0)
            [1.0,  "#00c896"],              # teal-green → large positive residual
        ],
        cmin=-resid.abs().quantile(0.97),   # symmetric colour axis at 97th pctile
        cmax= resid.abs().quantile(0.97),
        opacity=0.85,
        line=dict(width=0),
        colorbar=dict(
            title=dict(text="Residual ($)", font=dict(color="#aaccdd", size=11)),
            tickfont=dict(color="#aaccdd", size=10),
            x=1.15,
            len=0.6,
        ),
    ),
    name="Observed Sales",
    hovertemplate=(
        "<b>Observation</b><br>"
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} mi<br>"
        "Actual: $%{z:,.0f}<extra></extra>"
    ),
))

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 ── Layout, Axes, and Dark Theme
# ─────────────────────────────────────────────────────────────────────────────

axis_style = dict(
    backgroundcolor="#060f1a",
    gridcolor="#1a3a50",
    showbackground=True,
    zerolinecolor="#1a3a50",
    tickfont=dict(color="#8ab8cc", size=9, family="Courier New"),
    titlefont=dict(color="#aaccdd", size=12, family="Courier New"),
)

fig.update_layout(
    title=dict(
        text=(
            "<b style='font-family:Courier New; color:#00e5c8; font-size:18px'>"
            "OLS REGRESSION HYPERPLANE</b><br>"
            f"<span style='font-family:Courier New; color:#5588aa; font-size:12px'>"
            f"Ŷ = {beta_0:,.0f} "
            f"{'−' if beta_1 < 0 else '+'} {abs(beta_1):,.0f}·Age "
            f"{'−' if beta_2 < 0 else '+'} {abs(beta_2):,.0f}·Distance  "
            f"| R² = {results.rsquared:.4f}</span>"
        ),
        x=0.5,
        xanchor="center",
    ),
    scene=dict(
        xaxis=dict(**axis_style, title="Property Age (years)"),
        yaxis=dict(**axis_style, title="Distance to Tech Hub (mi)"),
        zaxis=dict(**axis_style, title="Sale Price ($)"),
        bgcolor="#060f1a",
        camera=dict(eye=dict(x=1.6, y=-1.8, z=0.9)),
        aspectmode="manual",
        aspectratio=dict(x=1.2, y=1.2, z=0.9),
    ),
    paper_bgcolor="#060f1a",
    plot_bgcolor="#060f1a",
    font=dict(family="Courier New", color="#aaccdd"),
    margin=dict(t=110, b=20, l=10, r=10),
    legend=dict(
        font=dict(color="#aaccdd", size=11),
        bgcolor="rgba(6,15,26,0.7)",
        bordercolor="#1a3a50",
        borderwidth=1,
        x=0.01, y=0.99,
    ),
    height=720,
)

fig.show()

# ─────────────────────────────────────────────────────────────────────────────
# OPTIONAL: Export as standalone HTML for portfolio / GitHub Pages
# ─────────────────────────────────────────────────────────────────────────────
# fig.write_html("ols_3d_hyperplane.html", include_plotlyjs="cdn")
# print("✅ Saved: ols_3d_hyperplane.html")

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.665
Model:                            OLS   Adj. R-squared:                  0.663
Method:                 Least Squares   F-statistic:                     295.2
Date:                Fri, 13 Mar 2026   Prob (F-statistic):           2.53e-71
Time:                        18:52:12   Log-Likelihood:                -3557.3
No. Observations:                 300   AIC:                             7121.
Df Residuals:                     297   BIC:                             7132.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 8.543e+05 